In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
history_path = Path("../logs/train_history.json")
with open(history_path) as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss = [h["val_loss"] for h in history]
train_f1 = [h["train_f1"] for h in history]
val_f1 = [h["val_f1"] for h in history]
lr = [h["lr"] for h in history]

best_epoch = int(np.argmax(val_f1)) + 1
best_val_f1 = max(val_f1)
print(f"Best epoch: {best_epoch}")
print(f"Best val F1: {best_val_f1:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_loss, label="Train loss", marker="o", markersize=3)
ax.plot(epochs, val_loss, label="Val loss", marker="o", markersize=3)
ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.5,
           label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_f1, label="Train F1", marker="o", markersize=3)
ax.plot(epochs, val_f1, label="Val F1", marker="o", markersize=3)
ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.5,
           label=f"Best epoch ({best_epoch})")
ax.scatter([best_epoch], [best_val_f1], color="red", s=100, zorder=5,
           label=f"Best val F1 = {best_val_f1:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Macro F1")
ax.set_title("Training and Validation Macro F1")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
final_gap = train_f1[-1] - val_f1[-1]
print(f"Final train F1: {train_f1[-1]:.4f}")
print(f"Final val F1:   {val_f1[-1]:.4f}")
print(f"Gap (overfitting): {final_gap:.4f}")
print()
print("Interpretation:")
print("- Small gap (<0.05) = underfit or well-regularized")
print("- Medium gap (0.05-0.15) = mild overfitting")
print("- Large gap (>0.15) = severe overfitting")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, lr, marker="o", markersize=3, color="purple")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("OneCycle Learning Rate Schedule")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
## Observations

1. **Best val F1 = 0.7084** at epoch 10
2. **Test macro F1 = 0.6895** — slightly lower, expected (validation was used for model selection)
3. **Overfitting is visible**: train F1 climbs to 0.92 while val F1 plateaus at 0.68
4. **The gap is ~0.23 at epoch 50** — confirms early stopping was correct
5. **Per-class AUC remains strong** (0.84–0.95) — the model ranks correctly, just needs threshold tuning for imbalanced classes

## Next steps

- Deployment story (ONNX + INT8) is more valuable than +0.02 F1
- Could try stronger regularization / focal loss in future work